# HPV Data Preparation — Merge FASTA Sequences into CSV

This notebook:
1. Mounts your **Google Drive** and reads all four files from it
2. Parses each FASTA and extracts accession → sequence pairs
3. Joins the sequences into the corresponding CSV on the `Accession` column
4. Saves the enriched CSVs back to your Google Drive

---
**Expected input files (place these in your Google Drive):**
- `nucleotides.csv` + `nucleotides.fasta`
- `proteins.csv` + `proteins.fasta`

**FASTA header format:** `>ACCESSION.VERSION | description`  
**CSV Accession format:** `ACCESSION` (without version suffix)  
The notebook strips the `.VERSION` suffix from FASTA headers to match CSV accessions.

🔗 **[Open Google Drive](https://drive.google.com)**

## 1. Install Dependencies

In [1]:
!pip install biopython --quiet
print('✅ Dependencies ready')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 25.1 MB/s eta 0:00:00
✅ Dependencies ready


## 2. Mount Google Drive

Running this cell will ask you to sign in to your Google account and grant Colab access to your Drive.

After mounting, your Drive is accessible at `/content/drive/MyDrive/`.

👉 **Edit `DRIVE_FOLDER`** below if your files are inside a subfolder (e.g. `HPV_Project`).

In [4]:
from google.colab import drive
drive.mount('/content/drive')

import os

# ─────────────────────────────────────────────────────────────────────────
# 📁 CONFIGURATION — change DRIVE_FOLDER if your files are in a subfolder
DATA_FOLDER = '/content/drive/MyDrive/Timski/data'
RESULTS_FOLDER = '/content/drive/MyDrive/Timski/results'

# ─────────────────────────────────────────────────────────────────────────

# Input paths
NUC_CSV    = os.path.join(DATA_FOLDER, 'nucleotides.csv')
NUC_FASTA  = os.path.join(DATA_FOLDER, 'nucleotides.fasta')
PROT_CSV   = os.path.join(DATA_FOLDER, 'proteins.csv')
PROT_FASTA = os.path.join(DATA_FOLDER, 'proteins.fasta')

# Output paths (same folder, new filenames)
NUC_OUT  = os.path.join(RESULTS_FOLDER, 'nucleotides_with_sequences.csv')
PROT_OUT = os.path.join(RESULTS_FOLDER, 'proteins_with_sequences.csv')

# Verify all input files exist before continuing
print('Checking input files...')
all_found = True
for path in [NUC_CSV, NUC_FASTA, PROT_CSV, PROT_FASTA]:
    exists = os.path.exists(path)
    icon = '✅' if exists else '❌  NOT FOUND'
    print(f'  {icon} — {path}')
    if not exists:
        all_found = False

if all_found:
    print('\n✅ All input files found — ready to proceed!')
else:
    print('\n❌ Some files are missing. Make sure they are in your Drive and that DRIVE_FOLDER is set correctly.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checking input files...
  ✅ — /content/drive/MyDrive/Timski/data/nucleotides.csv
  ✅ — /content/drive/MyDrive/Timski/data/nucleotides.fasta
  ✅ — /content/drive/MyDrive/Timski/data/proteins.csv
  ✅ — /content/drive/MyDrive/Timski/data/proteins.fasta

✅ All input files found — ready to proceed!


## 3. Helper Functions — Parse FASTA & Merge into CSV

In [5]:
import pandas as pd
from Bio import SeqIO
import re

def parse_fasta(fasta_path):
    """
    Parse a FASTA file into a dict: {accession_without_version: sequence_string}.
    Strips the .VERSION suffix (e.g. QEG53821.1 -> QEG53821) to match CSV accessions.
    """
    fasta_dict = {}
    duplicates = []

    with open(fasta_path, 'r') as f:
        for record in SeqIO.parse(f, 'fasta'):
            raw_id = record.id                          # e.g. 'QEG53821.1'
            accession = re.sub(r'\.\d+$', '', raw_id)  # -> 'QEG53821'
            sequence = str(record.seq)

            if accession in fasta_dict:
                duplicates.append(accession)
            fasta_dict[accession] = sequence  # last occurrence wins

    if duplicates:
        print(f'  ⚠️  {len(duplicates)} duplicate accessions found (kept last): {duplicates[:5]}')
    print(f'  ✅ Parsed {len(fasta_dict)} sequences from {os.path.basename(fasta_path)}')
    return fasta_dict


def merge_fasta_into_csv(csv_path, fasta_dict, label=''):
    """
    Read a CSV, add a 'Sequence' column by joining on the Accession column.
    Returns the enriched DataFrame.
    """
    df = pd.read_csv(csv_path)
    total_rows = len(df)

    df['Sequence'] = df['Accession'].map(fasta_dict)

    matched = df['Sequence'].notna().sum()
    unmatched = total_rows - matched

    print(f'  [{label}] Rows: {total_rows} | Matched: {matched} | Unmatched: {unmatched}')
    if unmatched > 0:
        missing = df[df['Sequence'].isna()]['Accession'].tolist()
        print(f'  ⚠️  Sample unmatched accessions: {missing[:5]}')

    return df

print('✅ Helper functions defined')

✅ Helper functions defined


## 4. Process Nucleotides

In [6]:
print('--- Nucleotides ---')
nuc_fasta_dict = parse_fasta(NUC_FASTA)
nuc_df = merge_fasta_into_csv(NUC_CSV, nuc_fasta_dict, label='nucleotides')
nuc_df.head(3)

--- Nucleotides ---
  ✅ Parsed 13161 sequences from nucleotides.fasta
  [nucleotides] Rows: 13161 | Matched: 13161 | Unmatched: 0


,Accession,Organism_Name,Species,Family,Genotype,Isolate,Length,Nuc_Completeness,Geo_Location,Country,Host,Tissue_Specimen_Source,Submitters,Collection_Date,Release_Date,Molecule_type,Sequence
0,MK484705,Human papillomavirus type 16,Alphapapillomavirus 9,Papillomaviridae,NaN,xuca1916,7912,complete,China,China,Homo sapiens,NaN,"Xu,H., Zhang,W.",2018,2019-08-26,dsDNA,ACTACAATAATTCATGTATAAAATTAAGGGCGTAACCGAAATCGGT...
1,X94741,Human papillomavirus type 16,Alphapapillomavirus 9,Papillomaviridae,NaN,squamous cell cervical carcinoma from a Japane...,3036,partial,NaN,NaN,NaN,NaN,"Bauer-Hofmann,R., Borghouts,C., Auvinen,E., Bo...",NaN,1996-01-09,dsDNA,CTGCAGTCCAGCCTACGCAACAGAGCAAGACCCTATCTCAAAAAAA...
2,OR770716,Human papillomavirus 16,Alphapapillomavirus 9,Papillomaviridae,NaN,sample-98,288,partial,China: Xinjiang,China,Homo sapiens,NaN,"Cheng,H., Pan,Z.",2023,2023-11-22,dsDNA,TATTATGTCCTACATCTGTGTTTAGCAGCAACGAAGTATCCTCTCC...


## 5. Process Proteins

In [7]:
print('--- Proteins ---')
prot_fasta_dict = parse_fasta(PROT_FASTA)
prot_df = merge_fasta_into_csv(PROT_CSV, prot_fasta_dict, label='proteins')
prot_df.head(3)

--- Proteins ---
  ✅ Parsed 31209 sequences from proteins.fasta
  [proteins] Rows: 31209 | Matched: 31209 | Unmatched: 0


,Accession,Protein,Organism_Name,Species,Family,Genotype,Isolate,Length,Geo_Location,Country,Host,Tissue_Specimen_Source,Submitters,Collection_Date,Release_Date,Molecule_type,Sequence
0,QEG53821,E1 protein,Human papillomavirus type 16,Alphapapillomavirus 9,Papillomaviridae,NaN,xuca1916,649,China,China,Homo sapiens,NaN,"Xu,H., Zhang,W.",2018,2019-08-26,dsDNA,MADPAGTNGEEGTGCNGWFYVEAVVEKKTGDAISDDENENDSDTGE...
1,QEG53822,E2 protein,Human papillomavirus type 16,Alphapapillomavirus 9,Papillomaviridae,NaN,xuca1916,365,China,China,Homo sapiens,NaN,"Xu,H., Zhang,W.",2018,2019-08-26,dsDNA,METLCQRLNVCQDKILTHYENDSTNLRDHIDYWKHMRLECAIYYKA...
2,QEG53823,E5 protein,Human papillomavirus type 16,Alphapapillomavirus 9,Papillomaviridae,NaN,xuca1916,83,China,China,Homo sapiens,NaN,"Xu,H., Zhang,W.",2018,2019-08-26,dsDNA,MTNLDTASTTLLACFLLCFCVLLCVCLLIRPLLLSVSTYTSLILLV...


## 6. Save Enriched CSVs to Google Drive

Outputs will be saved to the same Drive folder as your input files:
- `nucleotides_with_sequences.csv`
- `proteins_with_sequences.csv`

🔗 **[Open Google Drive](https://drive.google.com)** to find them.

In [9]:
nuc_df.to_csv(NUC_OUT, index=False)
prot_df.to_csv(PROT_OUT, index=False)

print(f'✅ Saved: {NUC_OUT}')
print(f'✅ Saved: {PROT_OUT}')
print('\n🔗 Open Google Drive to view: https://drive.google.com')

✅ Saved: /content/drive/MyDrive/Timski/results/nucleotides_with_sequences.csv
✅ Saved: /content/drive/MyDrive/Timski/results/proteins_with_sequences.csv

🔗 Open Google Drive to view: https://drive.google.com
